In [ ]:
import asyncio
import websockets
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyeeg as pe
import warnings
import os
import pickle
import nest_asyncio
from collections import deque
import threading
import http.server
import socketserver
import scipy.signal as sig_proc
import concurrent.futures
import traceback

nest_asyncio.apply()
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_cwd = os.path.abspath(os.getcwd())
_candidates = [_cwd] + [os.path.abspath(os.path.join(_cwd, *(['..'] * i))) for i in range(1, 6)]
PROJECT_ROOT = next((p for p in _candidates if os.path.isdir(os.path.join(p, "data", "recordings_clean", "gat_output_paper"))), _cwd)

GAT_DIR    = os.path.join(PROJECT_ROOT, "data", "recordings_clean", "gat_output_paper")
MODELS_DIR = os.path.join(PROJECT_ROOT, "data", "recordings_clean", "openbci_models")

GAT_MODEL_PATH  = os.path.join(GAT_DIR, "best_model.pth")
GAT_SCALER_PATH = os.path.join(GAT_DIR, "scaler_final.pkl")
SCALER_PATH     = os.path.join(MODELS_DIR, "scaler.pkl")
SVM_ARO_PATH    = os.path.join(MODELS_DIR, "svm_aro.pkl")
SVM_VAL_PATH    = os.path.join(MODELS_DIR, "svm_val.pkl")
MLP_ARO_PATH    = os.path.join(MODELS_DIR, "mlp_aro.pkl")
MLP_VAL_PATH    = os.path.join(MODELS_DIR, "mlp_val.pkl")

LOCAL_PORT = 65432
HTTP_PORT  = 8000

N_CH    = 16
N_FEATS = 10
WINDOW_SIZE    = 250
INFERENCE_STEP = 32
BUFFER_CAPACITY = int(5 / (INFERENCE_STEP / 125))

SOS_FILTER = sig_proc.butter(4, [4, 45], btype='bandpass', fs=125, output='sos')

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"GAT_MODEL:     {GAT_MODEL_PATH}")
print(f"GAT_SCALER:    {GAT_SCALER_PATH}")
print(f"SVM/MLP DIR:   {MODELS_DIR}")
print(f"Device:        {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. GAT MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, num_heads=4,
                 attn_dropout=0.0, residual=True):
        super().__init__()
        self.H = num_heads; self.d = out_features; self.residual = residual
        self.W = nn.Linear(in_features, num_heads * out_features, bias=False)
        self.a = nn.Parameter(torch.empty(num_heads, 2 * out_features))
        nn.init.xavier_uniform_(self.a.unsqueeze(0))
        self.leaky     = nn.LeakyReLU(0.2)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.bn        = nn.BatchNorm1d(out_features)
        if residual:
            self.res_proj = (nn.Linear(in_features, out_features, bias=False)
                             if in_features != out_features else nn.Identity())
    def forward(self, x):
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.H, self.d)
        hi, hj = h.unsqueeze(2), h.unsqueeze(1)
        pair = torch.cat([hi.expand(B,N,N,self.H,self.d),
                          hj.expand(B,N,N,self.H,self.d)], dim=-1)
        e = self.leaky((pair * self.a[None,None,None]).sum(-1))
        alpha = self.attn_drop(F.softmax(e, dim=2))
        out = torch.einsum("bqkh,bkhd->bqhd", alpha, h).mean(2)
        out = self.bn(out.reshape(B*N, self.d)).reshape(B, N, self.d)
        out = F.elu(out)
        if self.residual: out = out + self.res_proj(x)
        return out, alpha.permute(0,3,1,2)

class TaskHead(nn.Module):
    def __init__(self, in_dim, dense, head_dropout=0.6):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, dense), nn.LayerNorm(dense), nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(dense, dense // 2), nn.GELU(),
            nn.Dropout(head_dropout * 0.5),
            nn.Linear(dense // 2, 1),
        )
    def forward(self, x): return self.head(x.mean(dim=1))

class DeepGAT(nn.Module):
    def __init__(self, n_channels, in_feats, backbone_dims, dense_size,
                 num_heads=4, attn_dropout=0.0, head_dropout=0.0):
        super().__init__()
        d0 = backbone_dims[0]
        self.input_proj = nn.Linear(in_feats, d0)
        self.ch_embed = nn.Parameter(torch.randn(1, n_channels, d0) * 0.02)
        dims = [d0] + backbone_dims
        self.backbone = nn.ModuleList([
            GATLayer(dims[i], dims[i+1], num_heads, attn_dropout, True)
            for i in range(len(backbone_dims))
        ])
        self.head_aro = TaskHead(backbone_dims[-1], dense_size, head_dropout)
        self.head_val = TaskHead(backbone_dims[-1], dense_size, head_dropout)
    def forward(self, x):
        x = F.gelu(self.input_proj(x)) + self.ch_embed
        for layer in self.backbone:
            x, _ = layer(x)
        return torch.cat([self.head_aro(x), self.head_val(x)], dim=1)

# ─────────────────────────────────────────────────────────────────────────────
# 3. LOAD MODELS
# ─────────────────────────────────────────────────────────────────────────────
print("\nLoading models...")

gat_scaler = None
if os.path.exists(GAT_SCALER_PATH):
    with open(GAT_SCALER_PATH, 'rb') as f: gat_scaler = pickle.load(f)
    print(f"GAT scaler loaded  (n_features={gat_scaler.n_features_in_})")
else:
    print(f"GAT scaler NOT FOUND: {GAT_SCALER_PATH}")

svm_mlp_scaler = None
if os.path.exists(SCALER_PATH):
    with open(SCALER_PATH, 'rb') as f: svm_mlp_scaler = pickle.load(f)
    print(f"SVM/MLP scaler loaded  (n_features={svm_mlp_scaler.n_features_in_})")
else:
    print(f"SVM/MLP scaler NOT FOUND: {SCALER_PATH}")
    svm_mlp_scaler = gat_scaler

gat_model = DeepGAT(
    n_channels=N_CH, in_feats=N_FEATS,
    backbone_dims=[32, 32], dense_size=64, num_heads=4,
    attn_dropout=0.1, head_dropout=0.6
).to(DEVICE)
if os.path.exists(GAT_MODEL_PATH):
    gat_model.load_state_dict(torch.load(GAT_MODEL_PATH, map_location=DEVICE))
    gat_model.eval()
    print("GAT model loaded.")
else:
    print(f"GAT model NOT FOUND: {GAT_MODEL_PATH}")

svm_aro, svm_val = None, None
if os.path.exists(SVM_ARO_PATH) and os.path.exists(SVM_VAL_PATH):
    with open(SVM_ARO_PATH, 'rb') as f: svm_aro = pickle.load(f)
    with open(SVM_VAL_PATH, 'rb') as f: svm_val = pickle.load(f)
    print("SVM models loaded.")

mlp_aro, mlp_val = None, None
if os.path.exists(MLP_ARO_PATH) and os.path.exists(MLP_VAL_PATH):
    with open(MLP_ARO_PATH, 'rb') as f: mlp_aro = pickle.load(f)
    with open(MLP_VAL_PATH, 'rb') as f: mlp_val = pickle.load(f)
    print("MLP models loaded.")

ACTIVE_MODEL = "gat"
print(f"\nActive model: {ACTIVE_MODEL.upper()}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. FEATURE EXTRACTION & INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
def differential_entropy(sig, band_edges, fs):
    de, fft_full = [], np.fft.rfft(sig)
    freqs = np.fft.rfftfreq(len(sig), 1.0 / fs)
    for lo, hi in zip(band_edges[:-1], band_edges[1:]):
        mask = (freqs >= lo) & (freqs < hi)
        f = np.zeros_like(fft_full); f[mask] = fft_full[mask]
        var = np.var(np.fft.irfft(f, n=len(sig))) + 1e-10
        de.append(0.5 * np.log(2 * np.pi * np.e * var))
    return de

def extract_features(raw_window_data, sample_rate=125):
    band_edges = [4, 8, 12, 16, 25, 45]
    feats = []
    for ch in range(raw_window_data.shape[0]):
        s = raw_window_data[ch, :]
        bp = list(pe.bin_power(s, band_edges, sample_rate)[0])
        de = differential_entropy(s, band_edges, sample_rate)
        feats.extend(bp + de)
    return np.array(feats, dtype=np.float32)

def run_inference(features_vector, model_name):
    feat_flat = features_vector.reshape(1, -1)
    if model_name == "gat":
        sc = gat_scaler
        fs = sc.transform(feat_flat) if sc is not None else feat_flat
        tens = torch.tensor(fs.reshape(1, N_CH, N_FEATS), dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            probs = torch.sigmoid(gat_model(tens)).cpu().numpy()[0]
        return float(probs[0]), float(probs[1])
    elif model_name == "svm" and svm_aro is not None:
        sc = svm_mlp_scaler
        fs = sc.transform(feat_flat) if sc is not None else feat_flat
        return float(svm_aro.predict_proba(fs)[0,1]), float(svm_val.predict_proba(fs)[0,1])
    elif model_name == "mlp" and mlp_aro is not None:
        sc = svm_mlp_scaler
        fs = sc.transform(feat_flat) if sc is not None else feat_flat
        return float(mlp_aro.predict_proba(fs)[0,1]), float(mlp_val.predict_proba(fs)[0,1])
    else:
        return run_inference(features_vector, "gat")

# ─────────────────────────────────────────────────────────────────────────────
# 5. WEBSOCKET SERVER
# ─────────────────────────────────────────────────────────────────────────────
prediction_history = deque(maxlen=BUFFER_CAPACITY)
connected_clients  = set()
_streamer_clients  = set()
_paused = False
_inference_pending = False
_executor = concurrent.futures.ThreadPoolExecutor(max_workers=2)


async def _send_safe(ws, msg_str):
    """Send to one client; discard if it fails."""
    try:
        await asyncio.wait_for(ws.send(msg_str), timeout=0.5)
    except Exception:
        pass


async def broadcast_to_ui(msg_dict):
    """Fire-and-forget send to UI clients only. Never blocks caller."""
    ui = connected_clients - _streamer_clients
    if not ui:
        return
    msg_str = json.dumps(msg_dict)
    for c in list(ui):
        asyncio.create_task(_send_safe(c, msg_str))


async def broadcast_msg(msg_dict):
    """Broadcast control messages to all clients (rare, OK to await)."""
    if not connected_clients:
        return
    msg_str = json.dumps(msg_dict)
    for c in list(connected_clients):
        asyncio.create_task(_send_safe(c, msg_str))


def _do_inference_sync(window_data, model_name):
    """Runs in thread pool — extract features + model inference."""
    feats = extract_features(window_data, sample_rate=125)
    return run_inference(feats, model_name)


def _on_inference_done(future, model_name, loop):
    """Callback from thread pool — posts result back to event loop."""
    global _inference_pending
    try:
        aro, val = future.result()
        asyncio.run_coroutine_threadsafe(_deliver_prediction(aro, val, model_name), loop)
    except Exception as e:
        print(f"⚠️ Inference error ({model_name}): {e}")
        _inference_pending = False


async def _deliver_prediction(aro, val, model_name):
    """Runs on event loop — updates history and sends to UI."""
    global _inference_pending
    prediction_history.append((aro, val))
    avg_aro = sum(p[0] for p in prediction_history) / len(prediction_history)
    avg_val = sum(p[1] for p in prediction_history) / len(prediction_history)
    await broadcast_to_ui({
        "type":    "prediction",
        "arousal": float(avg_aro),
        "valence": float(avg_val),
        "model":   model_name,
    })
    _inference_pending = False


async def handler(websocket):
    global ACTIVE_MODEL, _paused, _inference_pending
    print("-> Client connected!")
    connected_clients.add(websocket)
    data_buffer         = np.zeros((N_CH, 0))
    samples_accumulated = 0
    zi_state            = None
    is_streamer         = False
    loop                = asyncio.get_running_loop()

    try:
        async for message in websocket:
            try:
                received_data = json.loads(message)

                # ── Control commands ──────────────────────────────────
                if isinstance(received_data, dict):
                    msg_type = received_data.get("type")

                    if msg_type == "cmd_change_model":
                        new_model = received_data.get("model", "gat").lower()
                        if new_model in ("gat", "svm", "mlp"):
                            ACTIVE_MODEL = new_model
                            prediction_history.clear()
                            print(f"🧠 Model switched to: {ACTIVE_MODEL.upper()}")
                            await broadcast_msg({"type": "model_changed", "model": ACTIVE_MODEL})
                        continue

                    if msg_type == "cmd_start_stream":
                        data_buffer = np.zeros((N_CH, 0))
                        samples_accumulated = 0
                        zi_state = None
                        _paused = False
                        prediction_history.clear()
                        await broadcast_msg(received_data)
                        continue

                    if msg_type == "cmd_pause_stream":
                        _paused = True
                        await broadcast_msg(received_data)
                        continue

                    if msg_type == "cmd_resume_stream":
                        _paused = False
                        await broadcast_msg(received_data)
                        continue

                    if msg_type in ["stream_info", "stream_end", "progress"]:
                        await broadcast_msg(received_data)
                        continue

                # ── Raw EEG signal ────────────────────────────────────
                if isinstance(received_data, list) and not _paused:
                    if not is_streamer:
                        is_streamer = True
                        _streamer_clients.add(websocket)
                        print("   (identified as streamer)")

                    chunk = np.array(received_data)
                    if chunk.ndim < 2 or chunk.shape[1] == 0:
                        continue

                    # CAR + bandpass filter
                    chunk_car = chunk - np.mean(chunk, axis=0)
                    if zi_state is None:
                        zi_init  = sig_proc.sosfilt_zi(SOS_FILTER)
                        zi_state = zi_init[:, np.newaxis, :] * chunk_car[:, 0][np.newaxis, :, np.newaxis]

                    filtered_chunk, zi_state = sig_proc.sosfilt(
                        SOS_FILTER, chunk_car, axis=-1, zi=zi_state)

                    # Forward signal to UI — fire-and-forget, never blocks
                    ui_chunk = np.clip(filtered_chunk, -150.0, 150.0)
                    await broadcast_to_ui({"type": "signal", "data": ui_chunk.tolist()})

                    # Accumulate for inference
                    data_buffer = np.concatenate((data_buffer, filtered_chunk), axis=1)
                    samples_accumulated += chunk.shape[1]
                    if data_buffer.shape[1] > WINDOW_SIZE:
                        data_buffer = data_buffer[:, -WINDOW_SIZE:]

                    # Launch inference in thread pool (skip if one pending)
                    if (data_buffer.shape[1] == WINDOW_SIZE
                            and samples_accumulated >= INFERENCE_STEP
                            and not _inference_pending):
                        samples_accumulated = 0
                        _inference_pending = True
                        model_snap = ACTIVE_MODEL
                        fut = _executor.submit(
                            _do_inference_sync, data_buffer.copy(), model_snap
                        )
                        fut.add_done_callback(
                            lambda f, m=model_snap, lo=loop: _on_inference_done(f, m, lo)
                        )

            except Exception as e:
                print(f"⚠️ Handler error: {e}")
    except websockets.exceptions.ConnectionClosed:
        print("<- Client disconnected.")
    finally:
        connected_clients.discard(websocket)
        _streamer_clients.discard(websocket)


# ─────────────────────────────────────────────────────────────────────────────
# 6. HTTP SERVER
# ─────────────────────────────────────────────────────────────────────────────
class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=".", **kwargs)
    def log_message(self, format, *args):
        pass

def run_http_server():
    with socketserver.TCPServer(("", HTTP_PORT), QuietHandler) as httpd:
        print(f"🌐 HTTP server: http://localhost:{HTTP_PORT}")
        httpd.serve_forever()


# ─────────────────────────────────────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────────────────────────────────────
async def main():
    threading.Thread(target=run_http_server, daemon=True).start()
    print(f"📡 WebSocket server on port {LOCAL_PORT}...")
    print(f"🧠 Models available: GAT" +
          (", SVM" if svm_aro else "") +
          (", MLP" if mlp_aro else ""))
    async with websockets.serve(handler, "0.0.0.0", LOCAL_PORT, ping_interval=None):
        await asyncio.Future()

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        print("\nStopped.")


PROJECT_ROOT:  c:\Users\PC\Desktop\EEG_GraphAttentionNetwork
GAT_MODEL:     c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\recordings_clean\gat_output_paper\best_model.pth
GAT_SCALER:    c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\recordings_clean\gat_output_paper\scaler_final.pkl
SVM/MLP DIR:   c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\recordings_clean\openbci_models
Device:        cuda

Loading models...
GAT scaler loaded  (n_features=10)
SVM/MLP scaler loaded  (n_features=160)
GAT model loaded.
SVM models loaded.
MLP models loaded.

Active model: GAT
📡 WebSocket server on port 65432...
🧠 Models available: GAT, SVM, MLP
🌐 HTTP server: http://localhost:8000
-> Client connected!
-> Client connected!
